# ۰۲ - پاک‌سازی و آماده‌سازی داده‌ها

این نوتبوک تصمیم‌های پاک‌سازی را روی داده واقعی اجرا و مستند می‌کند و ویژگی‌های زمانی و شاخص‌های تحویل را می‌سازد.

## ۲. سؤال‌های کسب‌وکاری

1. برای هر مقادیر گمشده چه تصمیمی گرفته شده و چرا؟
2. رکوردهای تکراری چگونه بررسی شدند؟
3. نوع داده‌ها چگونه اصلاح شد؟
4. چه ویژگی‌های جدیدی ساخته شد و با چه تعریفی؟
5. چه مقادیر غیرعادی شناسایی شدند؟

## ۱. عنوان و هدف

این نوتبوک تصمیم‌های پاک‌سازی را روی داده واقعی اجرا و مستند می‌کند و ویژگی‌های زمانی و شاخص‌های تحویل را می‌سازد.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from olist_analysis import metrics as M
from olist_analysis.config import FIGURES_DIR, REPORTS_DIR, TABLES_DIR
from olist_analysis.font_utils import persian_available
from olist_analysis.features import load_dataset
from olist_analysis.pipeline import load_headline

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ds = load_dataset()
headline = load_headline()
print("Persian font registered for charts:", persian_available())
print(f"orders={len(ds.orders):,}  items={len(ds.items):,}  "
      f"delivered={len(ds.delivered):,}  reviewed={len(ds.reviews_per_order):,}")


Persian font registered for charts: True
orders=99,441  items=112,650  delivered=96,470  reviewed=98,673


## ۴. روش محاسبه

خط لوله پاک‌سازی در `src/olist_analysis/cleaning.py` قرار دارد و کاملاً قابل بازتولید است: داده خام هرگز تغییر نمی‌کند و هیچ ردیفی حذف نمی‌شود.

In [2]:
display(pd.read_csv(TABLES_DIR / "02_cleaning_decisions.csv"))

,step,problem,evidence,options,decision,rationale,impact
0,timestamp_conversion,Timestamp columns are stored as text.,0 values failed strict parsing.,Keep as text / parse with datetime64.,Converted every timestamp column with the expl...,"All 99,441 orders and 112,650 item rows parse ...","Enables duration, delay and monthly trend calc..."
1,orders_missing_timestamps,order_approved_at / carrier / delivered timest...,"approved_at: 160, carrier: 1783, delivered: 29...",Impute / drop rows / keep NaT.,Keep NaT and filter per metric (delivery metri...,"The missingness is structural: canceled, unava...",No delivery metric is computed on an order tha...
2,review_duplicate_ids,review_id is declared as the primary key but i...,"789 ids repeat across 1603 rows, always with a...",Drop duplicates / keep the first / keep all an...,"Keep every row; identity becomes (review_id, o...","Each row belongs to a real, distinct order; de...",Review counts are reported per row; satisfacti...
3,review_multiple_per_order,Some orders carry more than one review row.,547 orders; 98673 orders remain after de-dupli...,Average the scores / keep the first / keep the...,Keep the latest review_answer_timestamp per or...,"Deterministic, does not invent an average, and...",Prevents score multiplication when joining rev...
4,review_comments_missing,review_comment_title and message are mostly em...,"title missing 87656 rows, message missing 5824...",Drop the columns / impute text / keep unused.,"Keep the columns, do not impute, and exclude t...",The quantitative satisfaction metrics rest on ...,No effect on any reported number.
5,products_missing_category,Some products have no category and no catalogu...,610 products of 32951 (also missing name/descr...,Drop products / impute a category / label unkn...,Label them 'unknown' and keep them in revenue ...,They have real sales; dropping them would make...,Category rankings exclude the 'unknown' bucket...
6,products_untranslated_categories,Two Portuguese categories have no English tran...,623 products affected.,Drop them / transliterate / fall back to the P...,Fall back to the Portuguese name and flag has_...,Loses no sales and keeps the category identifi...,Those two categories keep their original names...
7,geolocation_duplicates_and_outliers,The geolocation table has massive duplication ...,"1000163 rows, 261831 exact duplicates, 31 rows...",Join raw / drop duplicates only / build a cent...,"Filter to the bounding box, drop exact duplica...",One row per prefix keeps the join cardinality ...,Lookup shrinks to 19011 rows and can never mul...
8,geolocation_conflicting_state,A few zip prefixes map to more than one state ...,8 prefixes.,Drop / take the first / take the most frequent.,Take the modal state and keep n_states for tra...,State is never taken from geolocation; custome...,No reported geographic metric depends on the a...
9,payments_zero_installments,Two payment rows report zero installments with...,2 rows (both are payment_sequential = 2 fragme...,Delete / impute 1 / exclude from the installme...,"Keep the value in the order total, exclude the...",Deleting them would understate the paid amount...,"Installment statistics use 103,884 rows; payme..."


### شمارنده‌های پاک‌سازی (شواهد عددی هر تصمیم)

In [3]:
display(pd.read_csv(TABLES_DIR / "02_cleaning_summary.csv"))

,metric,value
0,customers_rows,99441
1,customers_unique_person,96096
2,geo_centroid_rows,19011
3,geo_distinct_prefixes,19015
4,geo_duplicate_rows,261831
5,geo_prefixes_conflicting_state,8
6,geo_prefixes_outside_bbox,20
7,geo_rows,1000163
8,geo_rows_outside_bbox,31
9,items_orders,98666


## ۵. بررسی کیفیت داده مرتبط

### بررسی ترتیب زمانی رویدادها

In [4]:
checks = {
    "approved قبل از purchase": ds.log["orders_approved_before_purchase"],
    "carrier قبل از approved": ds.log["orders_carrier_before_approved"],
    "delivered قبل از carrier": ds.log["orders_delivered_before_carrier"],
    "approved گمشده": ds.log["orders_missing_approved"],
    "delivered گمشده": ds.log["orders_missing_delivered"],
}
display(pd.Series(checks, name="count"))

approved قبل از purchase       0
carrier قبل از approved     1359
delivered قبل از carrier      23
approved گمشده               160
delivered گمشده             2965
Name: count, dtype: int64

### ویژگی‌های ساخته‌شده در سطح سفارش

In [5]:
cols = ["order_id", "purchase_ym", "purchase_weekday", "is_delivered",
        "delivery_days", "delay_days", "delivery_class", "delay_bucket",
        "item_count", "order_total_value", "review_score"]
display(ds.orders[cols].head(8))

,order_id,purchase_ym,purchase_weekday,is_delivered,delivery_days,delay_days,delivery_class,delay_bucket,item_count,order_total_value,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10,Monday,True,8.436574,-8,early,on_time_or_early,1.0,38.71,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07,Tuesday,True,13.782037,-6,early,on_time_or_early,1.0,141.46,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08,Wednesday,True,9.394213,-18,early,on_time_or_early,1.0,179.12,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11,Saturday,True,13.208750,-13,early,on_time_or_early,1.0,72.20,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02,Tuesday,True,2.873877,-10,early,on_time_or_early,1.0,28.62,5.0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07,Sunday,True,16.542245,-6,early,on_time_or_early,1.0,175.26,4.0
6,136cce7faa42fdb2cefd53fdc79a6098,2017-04,Tuesday,False,NaN,<NA>,NaN,NaN,1.0,65.95,2.0
7,6514b8ad8028c9f2cc2374ded245783f,2017-05,Tuesday,True,9.989826,-12,early,on_time_or_early,1.0,75.16,5.0


### تعریف طبقه‌بندی تحویل بر پایه تاریخ تقویمی

In [6]:
display(M.delivery_class_summary(ds))

,delivery_class,n_orders,pct_of_delivered,definition,avg_delay_days
0,early,88644,91.8876,delivered before the promised date,-13.71
1,on_time,1292,1.3393,delivered on the promised date,0.0
2,late,6534,6.7731,delivered after the promised date,10.62


**نکته مهم:** تاریخ تخمینی تحویل در داده با ساعت ۰۰:۰۰:۰۰ ذخیره شده است. اگر تأخیر با اختلاف زمان‌سنجی خام محاسبه شود، تقریباً همه سفارش‌ها «زودتر از موعد» طبقه‌بندی می‌شوند و تعداد سفارش‌های «دقیقاً به‌موقع» صفر می‌شود. بنابراین تأخیر بر پایه تفاضل **تاریخ تقویمی** تعریف شده است.

## ۸. یافته‌های واقعی

- صفر مقدار غیرقابل تبدیل در ستون‌های تاریخ (هیچ خطای پارس).
- ۲٬۹۶۵ سفارش تاریخ تحویل واقعی ندارند؛ این گمشدگی ساختاری است، نه خطا.
- ۷۸۹ شناسه نظر تکراری، ۱٬۶۰۳ سطر را پوشش می‌دهد؛ هر شناسه به سفارش‌های مختلف با امتیاز یکسان اشاره می‌کند.
- ۶۱۰ محصول دسته‌بندی ندارند (۱.۸۵ درصد) و با برچسب `unknown` نگه داشته می‌شوند.
- ۳۱ سطر مختصات خارج از مرزهای برزیل حذف شد و ۲۶۱٬۸۳۱ سطر تکراری ژئولوکیشن حذف شد.
- در نهایت هیچ ردیفی از هیچ جدول اصلی حذف نشد.

## ۱۰. محدودیت‌های تحلیل

امتیاز تحویل فقط برای سفارش‌های «تحویل‌شده» قابل محاسبه است؛ بنابراین درصدهای مربوط به تحویل، مخرجشان ۹۶٬۴۷۰ سفارش است و نه کل ۹۹٬۴۴۱ سفارش.

## ۱۱. نتیجه‌گیری

داده‌ها اکنون آماده تحلیل‌اند: انواع داده اصلاح شده، ویژگی‌های زمانی ساخته شده و همه تصمیم‌ها با شواهد عددی مستند شده‌اند.